In [1]:
import asyncio
import os
import ast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from colav_automaton import ColavAutomaton
from colav_automaton.classification import classify_unsafe_set_obstacles
from colav_automaton_evaluation.figure_generator import plot_xy_position_over_time
from hybrid_automaton import Automaton, RunResult, ContinuousState, AuxiliaryState
from riskenv import Agent, Obstacle

/home/ryan/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
ha: Automaton = ColavAutomaton(
    k_theta=1.0, 
    constant_velocity=2.0, 
    acceptance_radius=5, 
    los_distance_threshold=100, 
    longitudinal_offset_distance=50.0, 
    lateral_offset_distance=50.0
)
print (ha)

Hybrid Automaton Definition:
	name: COLAV Automaton
	id: 0
	initial_mode: Transit
	modes: [Transit, Fallback, Waypoint_Reached]
	transitions:
		Transit --[e1]--> Waypoint_Reached
		Transit --[e2]--> Fallback
		Transit --[e3]--> Transit
		Fallback --[e4]--> Transit
		Waypoint_Reached --[e5]--> Transit
	guards:
		e1: [waypoint_reached_guard]
		e2: [unsafe_conditions_guard]
		e3: [los_clear_to_waypoint_guard]
		e4: [safe_conditions_guard]
		e5: [virtual_waypoints_guard]
	resets:
		e1: <none>
		e2: <none>
		e3: generate_new_virtual_waypoint
		e4: <none>
		e5: pop_virtual_waypoint
	invariants:
		Transit: [<none>]
		Fallback: [<none>]
		Waypoint_Reached: [failing_invariant]
	continous_dynamics:
		Transit -> flow_los_heading
		Fallback -> constant_heading_dynamics
		Waypoint_Reached -> constant_heading_dynamics



In [3]:
print (repr(ha))

stateDiagram-v2
    direction LR
    [*] --> Transit
    Transit --> Waypoint_Reached: e1
    Transit --> Fallback: e2
    Transit --> Transit: e3
    Fallback --> Transit: e4
    Waypoint_Reached --> Transit: e5
    note right of Waypoint_Reached: invariant = failing_invariant


## Run a scenario

A single agent state, one goal waypoint, a static unsafe region, and one
mock AIS ship contact classified via `colav_automaton.classification`
(new in v1.0.4). The ship's aggregated `Maneuver` becomes the
`maneuver_bias` auxiliary state, which `generate_new_virtual_waypoint`
uses to override its default "easiest side" heuristic when the agent
has to route around the unsafe region.

In [ ]:
x0 = np.array([-200.0, -200.0, 0.0, 0.0, 0.0], dtype=float)

# NOTE: the 'waypoints' auxiliary state holds a *stack* whose front is the
# current target - one goal waypoint here, not a pre-planned multi-leg
# route. generate_new_virtual_waypoint pushes temporary avoidance
# waypoints ahead of this one on the fly; it doesn't take a route list.
goal_waypoint = [200.0, 200.0]

unsafe_region = [
    [20, 20],
    [-50, -50],
    [50, -50],
    [50, 100],
    [-50, 100],
]

AGENT_SAFETY_RADIUS = 30.0
DSF = 80.0                 # distance safety factor for classification's I1/I2/I3 filtering
TIME_OF_INTEREST = 30.0    # seconds, TCPA horizon

# Mock AIS ship - reciprocal course, dead ahead of the agent -> head-on.
ships = [
    {"tag": "cargo", "position": (-100.0, -200.0), "heading": np.pi, "speed": 2.0},
]

agent = Agent(
    position=tuple(x0[0:2]), heading=x0[2], speed=x0[3], yaw_rate=x0[4],
    safety_radius=AGENT_SAFETY_RADIUS,
)
obstacles = [
    Obstacle(
        position=s["position"], heading=s["heading"], speed=s["speed"],
        yaw_rate=0.0, safety_radius=15.0, tag=s["tag"],
    )
    for s in ships
]

maneuver = classify_unsafe_set_obstacles(agent=agent, obstacles=obstacles, dsf=DSF, time_of_interest=TIME_OF_INTEREST)
maneuver_bias = maneuver.as_bias()
print(f"encounter={maneuver.encounter.value} side={maneuver.side} give_way={maneuver.give_way} "
      f"urgency={maneuver.urgency:.2f}\nreason: {maneuver.reason}")

In [ ]:
LOG_DIR = os.path.join(os.getcwd(), "notebook-run-logs")

results: RunResult = await ha.activate(
    initial_continuous_state=ContinuousState(
        name="agent_state",
        x0=x0,
        x_labels=["x", "y", "theta", "velocity", "yaw_rate"],
    ),
    initial_auxiliary_states=[
        AuxiliaryState(name="waypoints", aux0=goal_waypoint, aux_buffer_len=10),
        AuxiliaryState(name="unsafe_region", aux0=unsafe_region, aux_buffer_len=10, expected_update_hz=10),
        AuxiliaryState(name="maneuver_bias", aux0=maneuver_bias, aux_buffer_len=10, expected_update_hz=10),
    ],
    delta_time=0.1,
    enable_real_time_mode=False,
    continuous_state_sampler_enabled=True,
    continuous_state_sampler_rate=100,
    enable_self_integration=True,
    auxiliary_states_sampler_enabled=True,
    auxiliary_states_sampler_rate=10,
    should_write_logs=True,
    output_dir=LOG_DIR,
)
print(results)

In [ ]:
# ha.activate(...) returns a run *summary* (status, termination, timing) -
# per-step trajectory data is written to CSV logs under
# results.run_logs_dir_path (since should_write_logs=True above), not held
# in memory. Read continuous_state.csv back into the (timestamp, state)
# tuple list plot_xy_position_over_time expects.
continuous_log = pd.read_csv(os.path.join(results.run_logs_dir_path, "continuous_state.csv"))
x = [
    (row.timestamp, np.array(ast.literal_eval(row.state)))
    for row in continuous_log.itertuples()
]
print(f"{len(x)} continuous-state samples loaded from {results.run_logs_dir_path}")

In [ ]:
# Reuses colav_automaton_evaluation's real plotting helper instead of the
# ad-hoc inline copy this cell used to carry.
fig = plot_xy_position_over_time(x, unsafe_region, waypoints=[goal_waypoint])
plt.show()